# 03 · Wstępne przetwarzanie sygnału EEG w MNE-Python

**Kurs:** Praktyczna Analiza Danych Eye-Tracking i EEG w Pythonie

Surowy zapis EEG zawiera nie tylko aktywność mózgu, ale też zakłócenia: szum sieci energetycznej, wolne dryfy, mrugnięcia, ruchy gałek ocznych, napięcie mięśni czy uszkodzone elektrody. W tym notebooku zbudujemy standardowy potok (*pipeline*) wstępnego przetwarzania:

1. wczytanie danych i praca z obiektem `mne.io.Raw`,
2. filtracja sieciowa (*notch*),
3. wykrywanie i interpolacja złych kanałów (PyPREP),
4. filtracja pasmowoprzepustowa 1–40 Hz,
5. re-referencja do średniej (*average reference*),
6. analiza składowych niezależnych (ICA) – usuwanie mrugnięć i artefaktów mięśniowych,
7. zapis oczyszczonych danych (oraz – dla chętnych – zapis w standardzie BIDS).

**Dane:** zestaw `sample` z MNE-Python – jednoczesny zapis MEG i EEG (60 elektrod) podczas prezentacji bodźców słuchowych i wzrokowych (Gramfort i in., 2013). Korzystamy tylko z kanałów EEG.

**Szacowany czas:** ok. 90–120 minut.

> ⏳ **Pierwsze uruchomienie:** zestaw `sample` waży ok. 1,6 GB. W Codespaces pobiera się automatycznie w tle podczas tworzenia środowiska. Jeśli jeszcze go nie ma, komórka z `data_path()` pobierze go sama – zajmie to kilka minut.

## 0. Przygotowanie środowiska

In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
from mne.preprocessing import ICA
from pyprep import NoisyChannels

KATALOG_GLOWNY = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
WYNIKI = KATALOG_GLOWNY / "wyniki"
WYNIKI.mkdir(exist_ok=True)

mne.set_log_level("WARNING")  # mniej komunikatów; ustaw "INFO", aby zobaczyć szczegóły obliczeń
mne.viz.set_browser_backend("matplotlib")  # przebiegi EEG jako statyczne wykresy w notebooku
plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.prop_cycle": plt.cycler(color=["#2a78d6", "#eb6834", "#1baf7a"])})
print("Wersja MNE-Python:", mne.__version__)

## 1. Wczytanie danych – obiekt `mne.io.Raw`

`mne.datasets.sample.data_path()` zwraca ścieżkę do zestawu `sample` (i w razie potrzeby go pobiera). Plik `.fif` wczytujemy funkcją `mne.io.read_raw_fif`. Parametr `preload=False` oznacza, że na razie wczytujemy tylko **nagłówek** – same próbki zostaną załadowane do pamięci dopiero wtedy, gdy będą potrzebne.

In [ ]:
katalog_sample = mne.datasets.sample.data_path()
plik_raw = katalog_sample / "MEG" / "sample" / "sample_audvis_raw.fif"
raw = mne.io.read_raw_fif(plik_raw, preload=False)
raw

Najważniejsze metadane przechowuje słownik `raw.info`:

In [ ]:
print(f"Częstotliwość próbkowania: {raw.info['sfreq']:.1f} Hz")
print(f"Długość nagrania: {raw.times[-1]:.1f} s")
print(f"Filtry sprzętowe: {raw.info['highpass']:.1f}–{raw.info['lowpass']:.1f} Hz")
print("Liczba kanałów według typu:", dict(Counter(raw.get_channel_types())))
print("Kanały oznaczone jako złe:", raw.info["bads"])

Zostawiamy tylko kanały EEG, kanał EOG (elektrookulogram – rejestruje ruchy oczu i mrugnięcia) oraz kanały wyzwalaczy (*stim*), w których zapisano momenty pojawienia się bodźców. Dopiero teraz ładujemy dane do pamięci.

In [ ]:
raw.pick(["eeg", "eog", "stim"]).load_data()
print("Po wyborze kanałów:", dict(Counter(raw.get_channel_types())))
print("Kanały oznaczone jako złe:", raw.info["bads"])

Pod spodem obiekt `Raw` to po prostu tablica NumPy o wymiarach *kanały × próbki*, zawsze w jednostkach **SI** (dla EEG – w woltach). Wyciągnijmy jedną sekundę z jednego kanału:

In [ ]:
dane, czasy = raw.get_data(picks="EEG 001", tmin=60, tmax=61, return_times=True)
print("Kształt tablicy:", dane.shape, "| jednostka: V")

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(czasy, dane[0] * 1e6)  # przeliczenie V → µV
ax.set(xlabel="Czas [s]", ylabel="Napięcie [µV]", title="EEG 001 – jedna sekunda surowego sygnału")
plt.show()

### Rozmieszczenie elektrod i przegląd sygnału

Kanały EEG w zestawie `sample` mają nazwy `EEG 001`–`EEG 060` (zamiast nazw systemu 10–20), ale ich położenie zostało zdigitalizowane, więc MNE wie, gdzie leży każda elektroda. Na czerwono zaznaczony jest kanał oznaczony jako zły.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 8.5))
raw.plot_sensors(show_names=True, axes=ax, show=False)
ax.set_title("Rozmieszczenie elektrod EEG (widok z góry, nos u góry)")
plt.show()

In [ ]:
fig = raw.plot(start=30, duration=10, n_channels=30, scalings=dict(eeg=60e-6, eog=200e-6, stim=1),
               show_scrollbars=False, show=False)
fig.suptitle("Surowe EEG – fragment 10 s (pierwsze 30 kanałów)", y=1.01, fontweight="bold")
plt.show()

Na surowym sygnale widać wolne „falowanie” (dryfy) i duże, powolne wychylenia w kanałach czołowych i w EOG – to **mrugnięcia**. Kanał `EEG 053`, oznaczony przez autorów nagrania jako zły, jest wyszarzony.

Widmo mocy (*Power Spectral Density*, PSD) pokazuje, jak moc sygnału rozkłada się na częstotliwości. Przerywanymi liniami zaznaczono 50 Hz (sieć energetyczna w Polsce i Europie) i 60 Hz (USA).

In [ ]:
widmo_surowe = raw.compute_psd(fmax=200, picks="eeg")
fig = widmo_surowe.plot(picks="eeg", exclude=[], amplitude=False, show=False)
for czestotliwosc, kolor in [(50, "gray"), (60, "black")]:
    fig.axes[0].axvline(czestotliwosc, color=kolor, ls="--", lw=1)
fig.axes[0].set_title("Widmo mocy surowego EEG (szara linia: kanał EEG 053)")
plt.show()

## 2. Filtracja sieciowa (*notch*)

Przewody elektryczne w budynku indukują w elektrodach sinusoidalne zakłócenia o częstotliwości sieci energetycznej: **50 Hz w Polsce i Europie**, **60 Hz w USA**, oraz ich harmonicznych (100, 150, … lub 120, 180, … Hz). Filtr *notch* („wycinający”) usuwa bardzo wąskie pasmo wokół tych częstotliwości.

Zestaw `sample` nagrano w Martinos Center (Massachusetts General Hospital, USA), więc w widmie widać niewielki pik przy **60 Hz**, a nie przy 50 Hz. Wniosek praktyczny: **zawsze najpierw sprawdzamy widmo, a dopiero potem dobieramy filtr.**

### Demonstracja: zakłócenie 50 Hz jak w polskim laboratorium

Aby zobaczyć filtr notch w akcji, dodajmy do *kopii* danych sztuczne zakłócenie 50 Hz o amplitudzie 15 µV i usuńmy je filtrem.

In [ ]:
demo = raw.copy().pick("eeg").crop(tmax=60)
demo.apply_function(lambda x: x + 15e-6 * np.sin(2 * np.pi * 50 * demo.times))
demo_po_filtrze = demo.copy().notch_filter(freqs=[50])

fig, ax = plt.subplots(figsize=(10, 4))
for dane_demo, opis in [(demo, "z zakłóceniem 50 Hz"), (demo_po_filtrze, "po filtrze notch 50 Hz")]:
    widmo = dane_demo.compute_psd(fmin=35, fmax=65)
    ax.plot(widmo.freqs, 10 * np.log10(widmo.get_data().mean(axis=0) * 1e12), label=opis)  # V²/Hz → dB(µV²/Hz)
ax.set(xlabel="Częstotliwość [Hz]", ylabel="Moc [dB µV²/Hz]", title="Działanie filtru notch (średnia z kanałów)")
ax.legend()
plt.show()

Teraz filtr dla prawdziwych danych. Częstotliwość sieci ustawiamy w jednym miejscu – w danych z polskiego laboratorium wystarczy zmienić ją na 50 Hz.

In [ ]:
CZESTOTLIWOSC_SIECI = 60  # ✏️ Hz; dane „sample” pochodzą z USA – dla nagrań z Polski ustaw 50

harmoniczne = CZESTOTLIWOSC_SIECI * np.arange(1, 4)  # np. 60, 120, 180 Hz
raw.notch_filter(freqs=harmoniczne, picks=["eeg", "eog"])
print("Usunięto zakłócenia o częstotliwościach:", harmoniczne, "Hz")

## 3. Złe kanały: automatyczna detekcja (PyPREP) i interpolacja

Zła elektroda (odklejona, z wysoką impedancją, uszkodzona) psuje nie tylko własny kanał: przy re-referencji do średniej jej szum „rozleje się” na wszystkie pozostałe kanały. Dlatego złe kanały wykrywamy i **interpolujemy** (odtwarzamy z sąsiednich elektrod) **przed** re-referencją.

Biblioteka **PyPREP** implementuje kryteria z potoku PREP (Bigdely-Shamlo i in., 2015), m.in.:

* `deviation` – nietypowo duża lub mała amplituda,
* `correlation` – słaba korelacja z innymi kanałami,
* `hf_noise` – dużo szumu wysokoczęstotliwościowego (> 50 Hz),
* `ransac` – kanał źle przewidywany na podstawie losowych podzbiorów innych kanałów,
* `psd` – nietypowe widmo mocy.

In [ ]:
detektor = NoisyChannels(raw.copy().pick("eeg"), random_state=42)
detektor.find_all_bads(ransac=True)
for kryterium, kanaly in detektor.get_bads(as_dict=True).items():
    if kanaly:
        print(f"{kryterium:20s} {sorted(kanaly)}")

Algorytm oznaczył sporo kanałów – głównie elektrody czołowe i skroniowe (kryterium `psd`). Zanim cokolwiek zinterpolujemy, **obejrzyjmy sygnał** (dla czytelności wyświetlamy kopię przefiltrowaną 1–40 Hz).

In [ ]:
podejrzane = sorted(set(detektor.get_bads()) - {"EEG 053"})
podglad = raw.copy().filter(1, 40, picks=["eeg", "eog"])
fig = podglad.plot(picks=podejrzane + ["EEG 053", "EEG 030", "EEG 045", "EOG 061"], start=25, duration=15,
                   scalings=dict(eeg=60e-6, eog=200e-6), show_scrollbars=False, show=False)
fig.suptitle("Kanały wskazane przez PyPREP, EEG 053, dwa kanały kontrolne (030, 045) i EOG", y=1.01,
             fontweight="bold")
plt.show()

pozostale = [kanal for kanal in podglad.copy().pick("eeg").ch_names if kanal != "EEG 053"]
print(f"Odchylenie standardowe w paśmie 1–40 Hz: EEG 053 = {podglad.get_data(picks='EEG 053').std() * 1e6:.1f} µV, "
      f"mediana pozostałych kanałów = {np.median(podglad.get_data(picks=pozostale).std(axis=1)) * 1e6:.1f} µV")

Interpretacja:

* Kanał **EEG 053** jest wyraźnie uszkodzony – jego sygnał jest prawie płaski (wielokrotnie mniejsza amplituda niż w innych kanałach).
* Pozostałe podejrzane kanały wyglądają poprawnie. Ich „nietypowość” wynika z **mrugnięć** (duże, wolne wychylenia w kanałach czołowych) i **napięcia mięśni** czoła i skroni. To fizjologia, a nie uszkodzone elektrody – takie artefakty lepiej usunąć za pomocą ICA niż niszczyć interpolacją całe kanały.

Algorytmy automatyczne **podpowiadają**, ale decyzję podejmuje badacz (i opisuje ją w raporcie). Interpolujemy więc tylko kanał EEG 053. Interpolacja metodą *spherical splines* wykorzystuje położenia elektrod.

In [ ]:
raw.info["bads"] = ["EEG 053"]  # ✏️ spróbuj też: detektor.get_bads()
raw.interpolate_bads(reset_bads=True)
print(f"Po interpolacji – odchylenie standardowe EEG 053: {raw.get_data(picks='EEG 053').std() * 1e6:.1f} µV")
print("Kanały oznaczone jako złe:", raw.info["bads"])

## 4. Filtracja pasmowoprzepustowa 1–40 Hz

* **Filtr górnoprzepustowy (1 Hz)** usuwa wolne dryfy (pot, ruchy kabli, polaryzacja elektrod). Odcięcie 1 Hz jest też zalecane przed ICA (Winkler i in., 2015). Uwaga: w analizach ERP często stosuje się łagodniejsze 0,1 Hz, bo filtr 1 Hz może zniekształcać wolne komponenty (np. P300).
* **Filtr dolnoprzepustowy (40 Hz)** usuwa szybkie zakłócenia (m.in. większość aktywności mięśniowej i resztki szumu sieciowego).

MNE domyślnie projektuje filtr **FIR o zerowej fazie**, który nie przesuwa sygnału w czasie. Obejrzyjmy jego charakterystykę: odpowiedź impulsową (w czasie) i odpowiedź częstotliwościową.

In [ ]:
filtr = mne.filter.create_filter(None, raw.info["sfreq"], l_freq=1.0, h_freq=40.0)
fig = mne.viz.plot_filter(filtr, raw.info["sfreq"], flim=(0.1, 150), plot=("time", "magnitude"),
                          title="Filtr pasmowoprzepustowy 1–40 Hz (FIR, zerowa faza)")

In [ ]:
raw_przed_pasmem = raw.copy()  # kopia sprzed filtracji pasmowej – przyda się do ICA
raw.filter(l_freq=1.0, h_freq=40.0, picks=["eeg", "eog"])

fig, ax = plt.subplots(figsize=(10, 4))
for widmo, opis in [(widmo_surowe, "surowe"), (raw.compute_psd(fmax=200, picks="eeg"), "po filtracji")]:
    ax.plot(widmo.freqs, 10 * np.log10(np.median(widmo.get_data(), axis=0) * 1e12), label=opis)
ax.axvspan(1, 40, color="gray", alpha=0.15, label="pasmo przepustowe")
ax.set(xscale="log", xlabel="Częstotliwość [Hz]", ylabel="Moc [dB µV²/Hz]",
       title="Widmo mocy przed i po filtracji (mediana z kanałów – odporna na pojedynczy zły kanał)")
ax.legend()
plt.show()

## 5. Re-referencja do średniej

Napięcie EEG to zawsze **różnica potencjałów** między elektrodą a elektrodą odniesienia (referencyjną). Wybór referencji zmienia kształt przebiegów i map topograficznych. Przy gęstym rozmieszczeniu elektrod (≥ 32) popularna jest **referencja do średniej**: od każdego kanału w każdej chwili odejmujemy średnią ze wszystkich kanałów EEG. Dzięki temu żadna pojedyncza elektroda nie jest uprzywilejowana.

Właśnie dlatego złe kanały trzeba było naprawić wcześniej – trafiłyby do średniej i zanieczyściły wszystkie kanały.

In [ ]:
raw.set_eeg_reference("average", projection=False)

srednia = raw.get_data(picks="eeg").mean(axis=0)
print(f"Po re-referencji średnia ze wszystkich kanałów EEG wynosi ~0 (maks. |średnia| = {np.abs(srednia).max():.1e} V)")

## 6. Analiza składowych niezależnych (ICA)

Każda elektroda rejestruje **mieszaninę** sygnałów z wielu źródeł: mózgu, oczu, mięśni. ICA (*Independent Component Analysis*) rozkłada wielokanałowy sygnał na składowe, które są możliwie **statystycznie niezależne**. Artefakty (mrugnięcia, mięśnie) mają zwykle charakterystyczne, niezależne od mózgu źródła, więc trafiają do osobnych składowych. Usuwamy je i składamy sygnał z powrotem.

**Ważny trik:** ICA dopasowujemy na kopii danych o **szerszym paśmie (1–100 Hz)**. Artefakty mięśniowe mają większość mocy powyżej 20 Hz – w danych obciętych do 40 Hz trudno je rozpoznać. Wyznaczone filtry przestrzenne można potem zastosować do danych 1–40 Hz, bo filtracja w czasie nie zmienia sposobu mieszania źródeł na elektrodach.

In [ ]:
raw_do_ica = raw_przed_pasmem.copy().filter(l_freq=1.0, h_freq=100.0, picks=["eeg", "eog"])
raw_do_ica.set_eeg_reference("average", projection=False)

ica = ICA(n_components=20, method="fastica", random_state=97, max_iter="auto")
ica.fit(raw_do_ica, picks="eeg")
ica

Każda składowa ma swoją **topografię** (jak bardzo „widzi” ją każda elektroda). Już na tym etapie można zgadywać: składowa z silnym polem na czole to zwykle mrugnięcia, a bardzo ogniskowe składowe na brzegach głowy często pochodzą od mięśni.

In [ ]:
fig = ica.plot_components()

### Mrugnięcia: korelacja z kanałem EOG

`find_bads_eog` szuka składowych, których przebieg czasowy najsilniej koreluje z kanałem EOG.

In [ ]:
eog_idx, eog_wyniki = ica.find_bads_eog(raw_do_ica, ch_name="EOG 061")
eog_idx = [int(i) for i in eog_idx]  # zwykłe liczby całkowite – czytelniejszy wydruk
print("Składowe związane z ruchami oczu i mrugnięciami:", eog_idx)
fig = ica.plot_scores(eog_wyniki, exclude=eog_idx, title="Korelacja składowych ICA z kanałem EOG")

Szczegóły składowej „mrugnięciowej”: topografia (silne pole czołowe), obraz kolejnych segmentów sygnału, widmo i wariancja. Przebiegi składowych na tle kanału EOG pokazują, że składowa „odpala” dokładnie w chwilach mrugnięć.

In [ ]:
figs = ica.plot_properties(raw_do_ica, picks=eog_idx)

In [ ]:
fig = ica.plot_sources(raw_do_ica, picks=list(range(6)), start=25, stop=40, show_scrollbars=False, show=False)
fig.suptitle("Przebiegi pierwszych składowych ICA (na dole: kanał EOG)", y=1.01, fontweight="bold")
plt.show()

### Artefakty mięśniowe

`find_bads_muscle` (kryteria Dharmaprani i in., 2016) szuka składowych, które jednocześnie:

1. mają **rosnące lub płaskie widmo** w zakresie 7–45 Hz (aktywność mózgu ma widmo malejące, ~1/f),
2. są zlokalizowane **na obrzeżach** głowy (mięśnie skroniowe, czołowe, karku),
3. mają **ogniskową**, mało gładką topografię.

In [ ]:
miesnie_idx, miesnie_wyniki = ica.find_bads_muscle(raw_do_ica)
miesnie_idx = [int(i) for i in miesnie_idx]
print("Składowe mięśniowe:", miesnie_idx)
fig = ica.plot_scores(miesnie_wyniki, exclude=miesnie_idx, title="Wskaźnik artefaktu mięśniowego")

In [ ]:
figs = ica.plot_properties(raw_do_ica, picks=miesnie_idx[:2])

Widmo składowych mięśniowych jest płaskie aż do wysokich częstotliwości, a topografia – mocno ogniskowa. Automatyczne kryterium jest dość liberalne (oznaczyło wiele drobnych składowych), dlatego w prawdziwej analizie każdą składową warto obejrzeć. Pomocne bywa też narzędzie ICLabel (pakiet `mne-icalabel`; Pion-Tonachini i in., 2019).

### Usunięcie artefaktów

Oznaczamy składowe do usunięcia i stosujemy ICA do danych 1–40 Hz (z tą samą referencją do średniej).

In [ ]:
ica.exclude = sorted(set(eog_idx) | set(miesnie_idx))
print(f"Usuwamy {len(ica.exclude)} z {ica.n_components_} składowych: {ica.exclude}")

raw_czyste = ica.apply(raw.copy())

### Czy się udało? Średnie mrugnięcie przed i po ICA

`create_eog_epochs` wykrywa mrugnięcia w kanale EOG i wycina okna wokół nich. Uśredniony sygnał EEG wokół mrugnięcia pokazuje, jak silny jest artefakt – po ICA powinien praktycznie zniknąć.

In [ ]:
mrugniecia_przed = mne.preprocessing.create_eog_epochs(raw, ch_name="EOG 061", baseline=(-0.5, -0.3))
mrugniecia_po = mne.preprocessing.create_eog_epochs(raw_czyste, ch_name="EOG 061", baseline=(-0.5, -0.3))
print(f"Liczba wykrytych mrugnięć: {len(mrugniecia_przed)}")

fig, osie = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
mrugniecia_przed.average().plot(picks="eeg", axes=osie[0], show=False)
mrugniecia_po.average().plot(picks="eeg", axes=osie[1], show=False)
osie[0].set_title("Średnie mrugnięcie – przed ICA")
osie[1].set_title("Średnie mrugnięcie – po ICA")
plt.show()

I ten sam fragment sygnału z mrugnięciami – przed i po usunięciu składowych:

In [ ]:
moment = mrugniecia_przed.events[0, 0] / raw.info["sfreq"] - raw.first_time  # pierwsze wykryte mrugnięcie [s]
for dane_eeg, tytul in [(raw, "Przed ICA"), (raw_czyste, "Po ICA")]:
    fig = dane_eeg.plot(start=max(moment - 2, 0), duration=10, n_channels=20, scalings=dict(eeg=60e-6, eog=200e-6),
                        show_scrollbars=False, show=False)
    fig.suptitle(tytul, y=1.01, fontweight="bold")
    plt.show()

## 7. Zapis wyników

Zapisujemy oczyszczone dane i rozwiązanie ICA. Z pliku `wyniki/03_sample_eeg_oczyszczone_raw.fif` skorzystamy w notebooku 04. Zgodnie z konwencją MNE nazwy plików z danymi ciągłymi kończą się na `raw.fif`, a pliki ICA – na `ica.fif` (inaczej MNE wyświetli ostrzeżenie).

In [ ]:
raw_czyste.save(WYNIKI / "03_sample_eeg_oczyszczone_raw.fif", overwrite=True)
ica.save(WYNIKI / "03_sample_eeg-ica.fif", overwrite=True)
print("Zapisano:", *sorted(p.name for p in WYNIKI.glob("03_*")), sep="\n  ")

## 8. Dla chętnych: zapis danych w standardzie BIDS

**BIDS** (*Brain Imaging Data Structure*) to standard organizacji danych neuroobrazowych: ustalone nazwy katalogów i plików oraz pliki opisowe (`.json`, `.tsv`). Dane w BIDS łatwo udostępnić, zarchiwizować i przetwarzać gotowymi narzędziami. Pakiet **MNE-BIDS** (Appelhoff i in., 2019) zapisuje i wczytuje takie zbiory.

Zapisujemy **oryginalne**, nieprzetworzone nagranie – w BIDS przechowuje się surowe dane, a wyniki przetwarzania trafiają do osobnego katalogu `derivatives/`. Zestaw `sample` zawiera MEG i EEG, więc MNE-BIDS zapisze go jako dane MEG (wraz z kanałami EEG).

In [ ]:
from mne_bids import BIDSPath, print_dir_tree, read_raw_bids, write_raw_bids

raw_oryginalny = mne.io.read_raw_fif(plik_raw, preload=False)
raw_oryginalny.info["line_freq"] = 60  # częstotliwość sieci (wymagana przez BIDS)
zdarzenia = mne.find_events(raw_oryginalny, stim_channel="STI 014")
kody_zdarzen = {"sluchowe/lewe": 1, "sluchowe/prawe": 2, "wzrokowe/lewe": 3, "wzrokowe/prawe": 4,
                "buzka": 5, "przycisk": 32}

sciezka_bids = BIDSPath(subject="01", session="01", task="audiowizualne", run="1", root=WYNIKI / "bids_sample")
write_raw_bids(raw_oryginalny, sciezka_bids, events=zdarzenia, event_id=kody_zdarzen, overwrite=True)
print_dir_tree(WYNIKI / "bids_sample")

In [ ]:
raw_z_bids = read_raw_bids(sciezka_bids)
print(raw_z_bids)
print("Zdarzenia odczytane z pliku events.tsv:", sorted(map(str, set(raw_z_bids.annotations.description))))

## Podsumowanie

* Obiekt `mne.io.Raw` przechowuje dane (w woltach) i metadane (`raw.info`): częstotliwość próbkowania, typy i położenia kanałów, złe kanały.
* Kolejność kroków ma znaczenie: **złe kanały naprawiamy przed re-referencją do średniej**, a ICA dopasowujemy na danych przefiltrowanych górnoprzepustowo (≥ 1 Hz).
* Filtr sieciowy dobieramy do kraju nagrania (50 Hz w Polsce, 60 Hz w USA) – po obejrzeniu widma.
* Automatyczne narzędzia (PyPREP, `find_bads_eog`, `find_bads_muscle`) przyspieszają pracę, ale ich wyniki trzeba **zweryfikować wzrokowo**.

## Zadania do samodzielnego wykonania

**Zadanie 1.** Usuń tylko składową mrugnięć (`ica.exclude = eog_idx`) i porównaj widmo mocy oczyszczonych danych z wersją, w której usunięto także składowe mięśniowe. W jakim zakresie częstotliwości widać różnicę?

**Zadanie 2.** Dopasuj ICA do danych **1–40 Hz** (zamiast 1–100 Hz). Ile składowych mięśniowych wykrywa teraz `find_bads_muscle` i jak wyglądają ich widma? Wyjaśnij różnicę.

**Zadanie 3.** Zinterpoluj wszystkie kanały wskazane przez PyPREP (`detektor.get_bads()`). Jak zmienia się sygnał w kanałach czołowych i czy ICA nadal znajduje składową mrugnięć?

**Zadanie 4.** Zmień liczbę składowych ICA (`n_components`) na 15 i 30. Czy składowa mrugnięć jest wykrywana tak samo? Ile czasu trwa dopasowanie?

In [ ]:
# Twoje rozwiązania:

## Literatura

* Appelhoff, S., Sanderson, M., Brooks, T. L., i in. (2019). MNE-BIDS: Organizing electrophysiological data into the BIDS format and facilitating their analysis. *Journal of Open Source Software, 4*(44), 1896.
* Bigdely-Shamlo, N., Mullen, T., Kothe, C., Su, K.-M., i Robbins, K. A. (2015). The PREP pipeline: Standardized preprocessing for large-scale EEG analysis. *Frontiers in Neuroinformatics, 9*, 16.
* Dharmaprani, D., Nguyen, H. K., Lewis, T. W., i in. (2016). A comparison of independent component analysis algorithms and measures to discriminate between EEG and artifact components. *38th Annual International Conference of the IEEE EMBC*, 825–828.
* Gramfort, A., Luessi, M., Larson, E., i in. (2013). MEG and EEG data analysis with MNE-Python. *Frontiers in Neuroscience, 7*, 267.
* Pion-Tonachini, L., Kreutz-Delgado, K., i Makeig, S. (2019). ICLabel: An automated electroencephalographic independent component classifier, dataset, and website. *NeuroImage, 198*, 181–197.
* Widmann, A., Schröger, E., i Maess, B. (2015). Digital filter design for electrophysiological data – a practical approach. *Journal of Neuroscience Methods, 250*, 34–46.
* Winkler, I., Debener, S., Müller, K.-R., i Tangermann, M. (2015). On the influence of high-pass filtering on ICA-based artifact reduction in EEG-ERP. *37th Annual International Conference of the IEEE EMBC*, 4101–4105.